# Predicción de precios de vivienda en California

El mercado inmobiliario de California es uno de los más complejos del mundo.
En este notebook uso datos reales de bloques censales del estado para construir
un modelo capaz de predecir el precio medio de una vivienda a partir de
características como la ubicación, el ingreso de la zona y la densidad poblacional.

El enfoque es regresión lineal — un modelo supervisado que aprende la relación
entre esas características y el precio, y nos dice con cuánta precisión puede
estimar ese valor para una vivienda que nunca antes vio.

## Carga de datos

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

scaler = StandardScaler()
model = LinearRegression()

#cargar dateset limpio, output del EDA
df = pd.read_csv('../data/processed/housing_features_engineered.csv')

print(f"Dimensiones: {df.shape[0]} filas * {df.shape[1]} columnas")
df.head()

Dimensiones: 19087 filas * 19 columnas


,housing_median_age,median_income,median_house_value,rooms_per_household,age_x_income,population_per_room,geo_cluster_1,geo_cluster_2,geo_cluster_3,geo_cluster_4,geo_cluster_5,geo_cluster_6,geo_cluster_7,geo_cluster_8,geo_cluster_9,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,41.0,8.3252,452600.0,6.984127,341.3332,0.365909,False,True,False,False,False,False,False,False,False,False,False,True,False
1,21.0,8.3014,358500.0,6.238137,174.3294,0.338217,False,True,False,False,False,False,False,False,False,False,False,True,False
2,52.0,7.2574,352100.0,8.288136,377.3848,0.338105,False,True,False,False,False,False,False,False,False,False,False,True,False
3,52.0,5.6431,341300.0,5.817352,293.4412,0.437991,False,True,False,False,False,False,False,False,False,False,False,True,False
4,52.0,3.8462,342200.0,6.281853,200.0024,0.347265,False,True,False,False,False,False,False,False,False,False,False,True,False


## Contexto: hallazgos clave del EDA y feature engineering

El analisis exploratorio y el pipeline de preparacion previos produjeron
las siguientes decisiones que impactan directamente el modelado:

- **Techo artificial de $500,001 eliminado** en el EDA — los precios
  recortados distorsionarian el modelo.
- **`median_income` sigue siendo el predictor principal** con correlacion 0.65.
- **Coordenadas geograficas discretizadas** mediante K-Means en 10 zonas —
  el modelo recibe clusters con significado real en vez de coordenadas crudas.
- **`age_x_income`** captura la interaccion entre antiguedad e ingreso
  (correlacion 0.53) — invisible para Pearson por separado.
- **`population_per_room`** reemplaza a `population_per_household`
  por mayor correlacion con el precio (-0.32 vs -0.26) y menor redundancia.

## Selección de features

In [30]:
#convertir ocean_proximity a variables numericas con one-hot encoding
df_encoded = pd.get_dummies(df,columns=['ocean_proximity'],drop_first=True)

features = [
    'median_income',            #correlacion 0.65 - predictor principal
    'rooms_per_household',      #correlacion 0.26 - espacio disponible por hogar
    'population_per_household', #correlacion -0.26 - densidad humana por hogar
    'latitude',                 #ubicacion geografica
    'longitude'                 #ubicacion geografica
] + [col for col in df_encoded.columns if col.startswith('ocean_proximity_')]

X = df_encoded[features]
y = df_encoded['median_house_value']

print("Features seleccionadas:")
print(X.columns.tolist())
print(f"\nDimensiones  X: {X.shape} | y: {y.shape}")

KeyError: "None of [Index(['ocean_proximity'], dtype='str')] are in the [columns]"

## División train/test

Se divide el dataset en 80% entrenamiento y 20% test.
El modelo aprende exclusivamente sobre el conjunto de entrenamiento —
el test permanece sellado hasta la evaluación final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      #20% para test
    random_state=42     #semilla fija - resultados reproducibles
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Test:          {X_test.shape[0]} filas")
print(f"Proporcion test: {X_test.shape[0] / X.shape[0]:.0%}")

Entrenamiento: 15269 filas
Test:          3818 filas
Proporcion test: 20%


## Escalado de features

Se estandarizan las features para que todas operen en la misma escala.
Esto es obligatorio para Ridge y Lasso, y mejora la interpretabilidad
de los coeficientes en la regresion lineal base.

El scaler se entrena exclusivamente con X_train para evitar data leakage.

In [ ]:
#entrenar el scaler solo con train y transformar
X_train_scaled = scaler.fit_transform(X_train)

#aplicar la misma transformacion a test sin re-entrenar
X_test_scaled = scaler.transform(X_test)

print("Media de cada feature en train (debe ser ~0 despues de escalar):")
print(X_train_scaled.mean(axis=0).round(6))
print("\nDesviacion estandar en train (debe ser ~1 despues de escalar):")
print(X_train_scaled.std(axis=0).round(6))

Media de cada feature en train (debe ser ~0 despues de escalar):
[ 0. -0. -0.  0. -0. -0.  0.  0. -0.  0. -0.  0.  0.  0.  0. -0. -0. -0.]

Desviacion estandar en train (debe ser ~1 despues de escalar):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Entrenamiento del modelo

In [ ]:
#entrenar modelo con OLS internamente
model.fit(X_train_scaled, y_train)

print("Modelo entrenado")
print(f"Intercepto (BO): {model.intercept_:.2f}")
print(f"Numero de coeficientes: {len(model.coef_)}")

Modelo entrenado
Intercepto (BO): 193789.36
Numero de coeficientes: 18


## Evaluacion del modelo

In [ ]:
#predecir sobre el conjunto de test
y_pred = model.predict(X_test_scaled)

#calcular metricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

RMSE: $58,059.43
R²:   0.6450
